# Make a Function Fit in Your Head

> 📓 *Companion to* **Modern Agentic AI Engineer** *· Ch 5 §5.1, §5.5 · type: drill*

A drill in **behavior-preserving refactoring**. You take the book's three-jobs-in-one
`handle()` method, pin its behavior with a *characterization test*, then split it into
one-job-per-level helpers — re-running the test after every step so you always know the
code still does what it did. You finish by learning to spot the smell profile of
AI-written code on sight.

## 🧠 Why this matters

A reader's working memory holds maybe **half a dozen things at once**. Every vague name,
every extra nesting level, every function doing two jobs occupies a slot — and when the
slots run out, comprehension fails and bugs slip through review.

Treat **cognitive load as a budget**: every line either *spends* from it or *refunds* to
it. "Clean" code is simply code that fits in a reader's head, and almost every rule in
Ch 5 falls out of that one constraint.

Now that AI writes most code, this matters *more*, not less: the model will happily emit
a 300-line function with five responsibilities and emit it again, differently, next week.
Whether your codebase converges toward coherence or entropy is decided by the standards
**you** hold it to.

## Objectives & prerequisites

**By the end you can:**

- Write a **characterization test** that pins down what code *currently does* before you
  touch it.
- Refactor a three-jobs-interleaved function into **one-job-per-level** helpers, re-running
  the test after each step so behavior never changes.
- Rename `n` / `data2` / `process` into **intent-revealing** names, and flatten nesting with
  **guard clauses**.
- Name the recognizable **smell profile of AI-generated code** on sight.

**Prereqs:** Ch 4 (typing, structure). Nothing beyond the stdlib here.

**Run first:** the Setup cell. It is fully **offline** — there is no model call in this
notebook at all, so `MOCK` only gates whether we *print* the live-mode note.

In [ ]:
# --- Setup -------------------------------------------------------------------
import os
import random
import textwrap

from dotenv import load_dotenv

load_dotenv()  # never hardcode keys; secrets come from the environment only

# This drill is pure local computation: no model is called. We keep the standard
# switch for consistency with the rest of the course. MOCK=1 (default) is offline.
MOCK = os.getenv("COMPANION_MOCK", "1") == "1"

random.seed(5)  # determinism for anything stochastic (none here, but it's the habit)

if MOCK:
    print("MOCK mode: this notebook is 100% offline. No API key needed, nothing billed.")
else:
    print("LIVE mode requested, but this drill makes no model calls -- nothing to bill.")

## 1 · The starting point: three jobs in one breath

Here is the book's first-draft agent step (§5.1). It works. It is also doing **three jobs
interleaved** at three different levels of abstraction: *fetching* history, *building* a
prompt string, *calling* the model, and *persisting* the reply — string-mashing and policy
tangled together.

To run it offline we give it a tiny **3-line fake `db` and `llm`** (no network, no SDK).
The fakes are deliberately dumb: an in-memory list and an echo. They exist only so the
function *runs*, which is what lets us pin its behavior.

In [ ]:
class FakeDB:
    """A 3-line stand-in for the database: an in-memory list of rows."""
    def __init__(self, rows): self.rows = rows
    async def fetch(self, _query, _thread_id): return self.rows
    async def execute(self, _query, _value): self.rows.append({"content": _value})


class FakeLLM:
    """A deterministic stand-in for the model: echoes a tag + the prompt length."""
    async def complete(self, prompt): return f"REPLY[len={len(prompt)}]"


class AgentBefore:
    """The book's BEFORE version: handle() does three jobs interleaved."""
    def __init__(self, db, llm):
        self.db = db
        self.llm = llm
        self.persona = "You are a helpful support agent."

    async def handle(self, msg):                 # before: three jobs interleaved
        text = msg["content"].strip().lower()
        rows = await self.db.fetch("SELECT ...", msg["thread_id"])
        history = [r["content"] for r in rows][-20:]
        prompt = self.persona + "\n" + "\n".join(history) + "\n" + text
        out = await self.llm.complete(prompt)
        await self.db.execute("INSERT ...", out)
        return out


print("Defined AgentBefore + fakes. Nothing run yet.")

## 2 · ⚠️ Pitfall: refactoring code that has *no* test

The single most common way refactoring goes wrong is doing it **blind** — editing code with
no test, then *hoping* behavior didn't change. It did; you just won't find out until prod.

The fix is a **characterization test**: not a test of what the code *should* do, but a
snapshot of what it *currently does*, bad behavior and all. It is a tripwire. As long as it
stays green, your refactor preserved behavior; the instant it goes red, you changed
something — and you know exactly which step did it.

We pin the exact reply string and the exact side effect (a row appended) for a fixed input.

In [ ]:
import asyncio


def fresh_world():
    """Build a known starting state so the test is deterministic."""
    db = FakeDB(rows=[{"content": "earlier message"}])
    return db, FakeLLM()


async def run_once(agent_cls):
    """Drive `handle` once against a fresh world; return (reply, rows_after)."""
    db, llm = fresh_world()
    agent = agent_cls(db, llm)
    reply = await agent.handle({"thread_id": "t1", "content": "  Hello THERE  "})
    return reply, db.rows


async def characterize(agent_cls):
    """Characterization test: pin the CURRENT observable behavior of agent_cls.

    Returns the (reply, rows) so we can eyeball it, and asserts the snapshot so any
    behavior change during refactoring trips the wire.
    """
    reply, rows = await run_once(agent_cls)
    # The snapshot we are pinning (captured from AgentBefore -- the source of truth):
    assert reply == "REPLY[len=60]", f"reply changed: {reply!r}"
    assert rows[-1] == {"content": "REPLY[len=60]"}, f"side effect changed: {rows[-1]!r}"
    assert len(rows) == 2, f"row count changed: {len(rows)}"
    return reply, rows


# `characterize` is async; in a notebook we await it at top level (no asyncio.run()).
reply, rows = await characterize(AgentBefore)
print("PINNED reply:", reply)
print("PINNED rows :", rows)
print("characterization test is GREEN for AgentBefore")

## 3 · 🔮 Predict: which refactor lowers load most?

We'll extract three seams, one per job, and re-run `characterize()` after each:

1. `recent(...)` — fetch + trim the history (the *fetch* job).
2. `build_prompt(...)` — assemble the prompt string (the *formatting* job).
3. `append(...)` — persist the reply (the *write* job).

**Predict before running the next cells:** of those three extractions, which one removes the
most from the reader's working-memory budget when they open `handle()` to fix a bug? (Hint:
which job is the most *distracting* string-plumbing that has nothing to do with the policy
`handle` is expressing?) Jot your guess, then watch how `handle` reads after each step.

### Step 1 — extract the *fetch* seam: `recent()`

One change at a time. We move history-fetching into a `Memory` collaborator and re-run the
test immediately. If it stays green, the move was behavior-preserving and we keep going.

In [ ]:
class Memory:
    """A small seam over the DB. Later this is where caching/compaction lives."""
    def __init__(self, db): self.db = db

    async def recent(self, thread_id, limit=20):
        rows = await self.db.fetch("SELECT ...", thread_id)
        return [r["content"] for r in rows][-limit:]

    async def append(self, thread_id, reply):
        await self.db.execute("INSERT ...", reply)


class AgentStep1:
    def __init__(self, db, llm):
        self.memory = Memory(db)
        self.llm = llm
        self.persona = "You are a helpful support agent."

    async def handle(self, msg):
        text = msg["content"].strip().lower()
        history = await self.memory.recent(msg["thread_id"], limit=20)  # <- seam 1
        prompt = self.persona + "\n" + "\n".join(history) + "\n" + text
        out = await self.llm.complete(prompt)
        await self.memory.append(msg["thread_id"], out)                 # (free: seam 3 too)
        return out


await characterize(AgentStep1)
print("Step 1 GREEN: behavior preserved after extracting recent()/append().")

### Step 2 — extract the *formatting* seam: `build_prompt()`

The prompt string-mashing is the most distracting plumbing in `handle` — and it's the one
thing you'll want to unit-test and version later (Ch 10). Pull it into a pure function. Pure
means: no `self`, no I/O, same inputs → same output. Pure functions are the cheapest things
in the world to test.

In [ ]:
def build_prompt(persona: str, history: list[str], user_text: str) -> str:
    """Pure function: assemble the model prompt. No I/O, trivially testable."""
    return persona + "\n" + "\n".join(history) + "\n" + user_text


class AgentStep2:
    def __init__(self, db, llm):
        self.memory = Memory(db)
        self.llm = llm
        self.persona = "You are a helpful support agent."

    async def handle(self, msg):
        text = msg["content"].strip().lower()
        history = await self.memory.recent(msg["thread_id"], limit=20)
        prompt = build_prompt(self.persona, history, text)             # <- seam 2 (pure)
        out = await self.llm.complete(prompt)
        await self.memory.append(msg["thread_id"], out)
        return out


await characterize(AgentStep2)
print("Step 2 GREEN: build_prompt() extracted, behavior identical.")

### Step 3 — the after: one level of abstraction, reads like a narrative

With the seams in place, `handle` collapses to the book's **after** form (§5.1). Each line is
now one decision a reviewer can check independently, and each helper is a place a test or a
replacement can attach. Same behavior — the test proves it — but it finally *fits in your
head*.

In [ ]:
from dataclasses import dataclass


@dataclass
class Message:
    """Collapse the 2-key dict into a named domain type (foreshadows §5.4)."""
    thread_id: str
    content: str


class Agent:
    def __init__(self, db, llm):
        self.memory = Memory(db)
        self.llm = llm
        self.persona = "You are a helpful support agent."

    async def handle(self, msg) -> str:           # after: one level of abstraction
        message = msg if isinstance(msg, Message) else Message(**msg)
        history = await self.memory.recent(message.thread_id, limit=20)
        prompt = build_prompt(self.persona, history, message.content.strip().lower())
        reply = await self.llm.complete(prompt)
        await self.memory.append(message.thread_id, reply)
        return reply


reply, rows = await characterize(Agent)
print("FINAL GREEN. handle() now reads as a 4-line narrative:")
print(textwrap.indent(
    "history = recent(...)\nprompt  = build_prompt(...)\nreply   = llm.complete(...)\nappend(reply)",
    "    "))

**What you just saw.** The characterization test stayed green through every step, so we
*know* behavior never changed — we only moved it. The biggest load drop came from extracting
`build_prompt`: it evicted the most attention-hungry plumbing (string-mashing) out of the
policy `handle` expresses. Extraction is highest-leverage where the distraction is highest.

## 4 · Naming drill: from `n`/`data2`/`process` to intent

Names are the densest documentation you'll ever write. A good name states **intent** — what
the thing means in the domain, not how it's implemented. Two rules from §5.1:

- The bigger a name's scope, the more descriptive it must be (a loop index can be `i`; a
  module constant cannot).
- If a name is *hard* to choose, that's design feedback — the thing probably has no single
  identity and wants to be split.

Below is a function drowning in vague names and deep nesting. Read it, then run the cell to
compare it against the renamed-and-flattened version (guard clauses instead of the nested
`if`).

In [ ]:
# BEFORE: vague names (n, data2, process), nesting wraps the whole body.
def process(data, n):
    result = []
    if data:
        for x in data:
            if x["score"] is not None:
                if x["score"] >= n:
                    result.append(x["id"])
    return result


# AFTER: intent-revealing names + guard clauses to flatten nesting.
def select_relevant_chunk_ids(chunks, min_score):
    if not chunks:                     # guard clause, not a wrapping if
        return []
    relevant = []
    for chunk in chunks:
        if chunk["score"] is None:     # guard: skip, don't nest deeper
            continue
        if chunk["score"] >= min_score:
            relevant.append(chunk["id"])
    return relevant


sample = [{"id": "a", "score": 0.9}, {"id": "b", "score": None}, {"id": "c", "score": 0.2}]

# Same behavior, dramatically lower load. Verify they agree:
assert process(sample, 0.5) == select_relevant_chunk_ids(sample, 0.5) == ["a"]
print("before==after:", select_relevant_chunk_ids(sample, 0.5))
print("max nesting depth: 3 (before) -> 1 (after); name now states the job")

### Collapse a long argument list into a dataclass

Three-or-fewer arguments is the rule; past that, related arguments are really **one concept
wearing a trench coat**. Bundle them. The call site stops being a guessing game about
positional order, and the type checker starts helping you.

In [ ]:
from dataclasses import dataclass


# BEFORE: 4 positional args -- which is which at the call site?
def make_request(model, max_tokens, temperature, top_p):
    return f"{model} maxtok={max_tokens} temp={temperature} top_p={top_p}"


# AFTER: one named concept. Defaults documented in one place; calls self-explain.
@dataclass
class SamplingConfig:
    model: str
    max_tokens: int = 1024
    temperature: float = 0.7
    top_p: float = 1.0


def make_request_v2(cfg: SamplingConfig) -> str:
    return f"{cfg.model} maxtok={cfg.max_tokens} temp={cfg.temperature} top_p={cfg.top_p}"


before = make_request("claude-opus-4-8", 1024, 0.7, 1.0)
after = make_request_v2(SamplingConfig(model="claude-opus-4-8"))
assert before == after
print(after)
print("call site went from 4 mystery positions to 1 self-documenting object")

## 5 · Spot-the-smell gallery: the AI-codebase profile

Alongside the classics — duplication, long functions, god classes — AI-era code has its own
recognizable smell profile (§5.5). The gallery below holds five offenders, each as a short,
*real-looking* snippet. Read each one and name the smell **before** revealing the diagnosis in
the next cell.

In [ ]:
GALLERY = {
    "A": textwrap.dedent('''
        # scattered across ten files, no single source of truth:
        prompt = "You are helpful.\n" + ctx + "\nUser: " + q          # file: router.py
        prompt = "You are an expert.\n" + history + user_msg           # file: worker.py
        prompt = SYSTEM + "\n\n" + "\n".join(turns) + "\n" + latest  # file: api.py
    ''').strip(),
    "B": textwrap.dedent('''
        def step(state: dict) -> dict:        # state grows keys nobody dares remove
            state["tmp"] = state.get("tmp", []) + [state["x"]]
            state["flag2"] = True
            return state
    ''').strip(),
    "C": textwrap.dedent('''
        DATA_PATH = "/Users/me/Desktop/run3/out.json"   # hardcoded absolute path
        print("DEBUG got here")                          # leftover print debugging
        # only works if you called load() in the cell above first
    ''').strip(),
    "D": textwrap.dedent('''
        def get_user(uid):
            try:
                return db.fetch(uid)
            except Exception:
                return None        # failure hidden today, wrong answer next week
    ''').strip(),
    "E": textwrap.dedent('''
        def retry_a(f): ...   # in payments.py
        def with_retry(f): ...# in fetch.py, subtly different backoff
        def _retry(f): ...    # in tools.py, third variant of the same idea
    ''').strip(),
}

for key, snippet in GALLERY.items():
    print(f"--- Exhibit {key} " + "-" * 40)
    print(snippet)
    print()

### 🔮 Predict, then reveal

Name the smell in each exhibit (A–E) before running the next cell. The diagnoses use the
book's exact vocabulary.

In [ ]:
DIAGNOSIS = {
    "A": ("Prompt spaghetti -- prompt strings concatenated ad hoc across files, no "
          "versioning, no single place to read what the model is told. Fix: named, "
          "versioned templates in one registry (Ch 10)."),
    "B": ("The dict of everything -- an untyped dict threaded through the pipeline, "
          "growing keys nobody dares remove. Fix: Pydantic models at every boundary (Ch 15)."),
    "C": ("Notebook residue -- hardcoded paths, print debugging, cell-order dependency "
          "now frozen into a function. Fix: parameters, real logging, explicit inputs."),
    "D": ("Defensive try/except wallpaper -- `except Exception: return None` hides the "
          "failure now and yields an unexplainable wrong answer later. Fix: catch specific "
          "errors, or let it raise."),
    "E": ("Abstraction mismatch -- three slightly-different retry helpers, a collective "
          "maintenance tax. The model didn't know one already existed; reviewers must. "
          "Fix: one shared retry utility."),
}

for key in GALLERY:
    print(f"Exhibit {key}: {DIAGNOSIS[key]}")
    print()

## 🎯 Senior lens: structure is an instruction to your AI assistant

The reason this drill pays off twice: **structure is the highest-leverage instruction you
give an AI coding assistant.** Models imitate the surrounding code. In a codebase where
`handle` is a clean 4-line narrative and rules live in domain objects, generated code follows
that shape. In a codebase where handlers are 200-line rule-soups, the model amplifies the
soup — confidently, and at scale.

So refactoring is no longer only for human readers. When you extract `build_prompt`, name
`select_relevant_chunk_ids`, and collapse args into a `SamplingConfig`, you are **curating the
distribution your tools sample from**. Clean domains beget clean generations. The senior move
is to invest in structure *early*, while the codebase is small enough that every later
generation inherits it.

## Recap

- **Cognitive load is a budget** (~6 slots). Vague names, deep nesting, and two-job functions
  all spend from it; clean code refunds it.
- **Characterization test first.** Pin current behavior, *then* refactor — re-running the test
  after each small step so any behavior change trips the wire immediately.
- **One job per level of abstraction.** Extracting `recent` / `build_prompt` / `append` turned
  a tangled `handle` into a navigable 4-line narrative without changing behavior.
- **Names state intent; guard clauses flatten nesting; long arg lists become dataclasses.**
- **AI-code smell profile:** prompt spaghetti, the dict-of-everything, notebook residue,
  try/except wallpaper, duplicated helpers — review for these by name.
- **Structure is an instruction to the model**: clean surroundings produce clean generations.

## Exercises

Use the empty cells below. (Solutions land in `solutions/` in Phase 2.)

1. **Break it on purpose.** In `Agent.handle`, change `limit=20` to `limit=10` and re-run
   `characterize(Agent)`. Predict *before* running: does the test go red? Why does that prove
   the characterization test is doing its job — and what does the failure message tell you?
2. **Extract a new seam.** The `.strip().lower()` normalization is a tiny job hiding in
   `handle`. Extract it into a pure `normalize(text)` function, wire it in, and confirm
   `characterize(Agent)` stays green. Predict whether the pinned reply length changes.
3. **Name the smell.** Write a 6–10 line snippet that mixes *two* AI smells from the gallery
   (e.g. a dict-of-everything threaded through try/except wallpaper). Then write the one-line
   diagnosis and the fix for each, in the book's vocabulary.

In [ ]:
# Exercise 1 -- your code here

In [ ]:
# Exercise 2 -- your code here

In [ ]:
# Exercise 3 -- your code here

## Next

You made one function fit in your head. Next you'll make the whole provider layer *swappable*
by turning these refactors into permanent **seams**.

- ▶️ **Next notebook:** [`05-02-patterns-as-seams.ipynb`](./05-02-patterns-as-seams.ipynb) —
  the five patterns (Protocol, Adapter, Factory, DI, Strategy, Repository) as typed boundaries,
  plus a domain rule that lives with its data.
- 🎓 **Capstone:** the seams you practiced here become real in
  [`../../../capstone/`](../../../capstone/) — `providers/` (Protocol + adapter + factory) and
  `domain/` (`Thread`/`Message`/`Run` owning their rules). This chapter is the 🔧 Build that
  skeleton starts from.
- 📖 **Book:** see §5.1 (naming, functions, cognitive load) and §5.5 (refactoring safely; the
  smells of AI codebases).